# PCA

## Principle Components

So how can you find the principal components of a training set? Luckily, there is
a standard matrix factorization technique called singular value decomposition (SVD)
that can decompose the training set matrix X into the matrix multiplication of three
matrices U Σ V⊺, where V contains the unit vectors that define all the principal
components that you are looking for, in the correct order.

In [1]:
import numpy as np

# X = np.random.randint(1, 9, (5, 3))
X = np.array([
    [2, 5, 6],
    [3, 2, 2],
    [5, 1, 9],
    [3, 9, 8],
    [3, 2, 4]
])
X_centered = X - X.mean(axis=0)

X_centered

array([[-1.2,  1.2,  0.2],
       [-0.2, -1.8, -3.8],
       [ 1.8, -2.8,  3.2],
       [-0.2,  5.2,  2.2],
       [-0.2, -1.8, -1.8]])

In [2]:
U, s, Vt = np.linalg.svd(X_centered)
c1 = Vt[0]
c2 = Vt[1]
c1, c2

(array([ 0.04815873, -0.83354534, -0.55034799]),
 array([-0.36344149,  0.49859498, -0.78696463]))

## Projecting down to d dimensions  
To project the training set onto the hyperplane and obtain a reduced dataset Xd-proj
 of
dimensionality d, compute the matrix multiplication of the training set matrix X by
the matrix Wd
, defined as the matrix containing the first d columns of V  
Xd‐proj = X Wd  
The following Python code projects the training set onto the plane defined by the first
two principal components:

In [3]:
W2 = Vt[:2].T
X2D = X_centered @ W2

In [4]:
X2D

array([[-1.16811447,  0.87705084],
       [ 3.58207222,  2.16568292],
       [ 0.65949909, -4.56854744],
       [-5.55483307,  0.93406002],
       [ 2.48137624,  0.59175366]])

## Using Sklearn

In [5]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X2D = pca.fit_transform(X)

## Explained Variance Ratio

In [6]:
pca.explained_variance_ratio_

array([0.64233918, 0.3427073 ])

## Choosing the right number of dimensions
The following code loads and splits the MNIST dataset (introduced in Chapter 3)
and performs PCA without reducing dimensionality, then computes the minimum
number of dimensions required to preserve 95% of the training sets variance:

In [7]:
from sklearn.datasets import fetch_openml

mnist = fetch_openml('mnist_784', as_frame=False)

X_train, y_train = mnist.data[:60_000], mnist.target[:60_000]
X_test, y_test = mnist.data[60_000:], mnist.target[60_000:]

pca = PCA()
pca.fit(X_train)
cumsum = np.cumsum(pca.explained_variance_ratio_)
d = np.argmax(cumsum >= 0.95) + 1
d

np.int64(154)

You could then set n_components=d and run PCA again, but theres a better option.
Instead of specifying the number of principal components you want to preserve,
you can set n_components to be a float between 0.0 and 1.0, indicating the ratio of
variance you wish to preserve:

In [8]:
pca = PCA(n_components=0.95)
X_reduced = pca.fit_transform(X_train)
pca.n_components_

np.int64(154)

Lastly, if you are using dimensionality reduction as a preprocessing step for a super
vised learning task (e.g., classification), then you can tune the number of dimensions
as you would any other hyperparameter (see Chapter 2). For example, the following
code example creates a two-step pipeline, first reducing dimensionality using PCA,
then classifying using a random forest. Next, it uses RandomizedSearchCV to find
a good combination of hyperparameters for both PCA and the random forest classi
fier. This example does a quick search, tuning only 2 hyperparameters, training on
just 1,000 instances, and running for just 10 iterations, but feel free to do a more
thorough search if you have the time:

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import make_pipeline

clf = make_pipeline(PCA(random_state=42),
                    RandomForestClassifier(random_state=42))

param_dist = {
    "pca__n_components": np.arange(10, 80),
    "randomforestclassifier__n_estimators": np.arange(50, 500)
}

rnd_search = RandomizedSearchCV(clf, param_dist, n_iter=20, cv=3, random_state=42)
rnd_search.fit(X_train[:1000], y_train[:1000])

,estimator,Pipeline(step...m_state=42))])
,param_distributions,"{'pca__n_components': array([10, 11... 78, 79]), 'randomforestclassifier__n_estimators': array([ 50, ...97, 498, 499])}"
,n_iter,20
,scoring,None
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,nan


In [10]:
rnd_search.best_params_

{'randomforestclassifier__n_estimators': np.int64(272),
 'pca__n_components': np.int64(28)}

## PCA for compression
Xrecovered = Xd‐projection Wd⊺

In [11]:
X_recovered = pca.inverse_transform(X_reduced)
X_recovered

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(60000, 784))

## Randomized PCA

In [12]:
rnd_pca = PCA(n_components=154, svd_solver="randomized", random_state=42)
X_reduced = rnd_pca.fit_transform(X_train)

## Incremental PCA 

In [13]:
from sklearn.decomposition import IncrementalPCA

n_batches = 100
inc_pca = IncrementalPCA(n_components=154)
for X_batch in np.array_split(X_train, n_batches):
    inc_pca.partial_fit(X_batch)

X_reduced = inc_pca.transform(X_train)

Alternatively, you can use NumPys memmap class, which allows you to manipulate a
large array stored in a binary file on disk as if it were entirely in memory; the class
loads only the data it needs in memory, when it needs it. To demonstrate this, lets
first create a memory-mapped (memmap) file and copy the MNIST training set to it,
then call flush() to ensure that any data still in the cache gets saved to disk. In real
life, X_train would typically not fit in memory, so you would load it chunk by chunk
and save each chunk to the right part of the memmap array:

In [14]:
filename = "my_mnist.mmap"
X_mmap = np.memmap(filename, dtype='float32', mode='write', shape=X_train.shape)
X_mmap[:] = X_train
X_mmap.flush()

Next, we can load the memmap file and use it like a regular NumPy array. Lets use
the IncrementalPCA class to reduce its dimensionality. Since this algorithm uses only
a small part of the array at any given time, memory usage remains under control.
This makes it possible to call the usual fit() method instead of partial_fit(),
which is quite convenient:

In [15]:
X_mmap = np.memmap(filename, dtype="float32", mode="readonly").reshape(-1, 784)
batch_size = X_mmap.shape[0]
inc_pca = IncrementalPCA(n_components=154, batch_size=batch_size)
inc_pca.fit(X_mmap)

,n_components,154
,whiten,False
,copy,True
,batch_size,60000


# Random Projection

In [11]:
import numpy as np
from sklearn.random_projection import johnson_lindenstrauss_min_dim
m, eps = 5_000, 0.1
d = johnson_lindenstrauss_min_dim(m, eps=eps)
d

np.int64(7300)

In [12]:
n = 20_000
np.random.seed(42)
P = np.random.randn(d, n) / np.sqrt(d)

X = np.random.randn(m, n)
X_reduced = X @ P.T

In [13]:
from sklearn.random_projection import GaussianRandomProjection

gaussian_rnd_proj = GaussianRandomProjection(eps=eps, random_state=42)
X_reduced = gaussian_rnd_proj.fit_transform(X)

In [14]:
from sklearn.random_projection import SparseRandomProjection

sparse_rnd_proj = SparseRandomProjection(eps=eps, random_state=42)
X_reduced = sparse_rnd_proj.fit_transform(X)

# LLE